<a href="https://colab.research.google.com/github/mafgit/developershub-internship-tasks/blob/master/Phase_2_Task_1_News_Topic_Classifier_Using_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset contains:
- title
- description
- label:
    1. World
    2. Sports
    3. Business
    4. Sci/Tech

In [ ]:
from datasets import load_dataset

dataset_name = "sh0416/ag_news"
dataset = load_dataset(dataset_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/33.7M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
print(dataset.keys(), '\n')

print(dataset['train'][0], '\n')

print(len(dataset['train']), '\n')

print(len(dataset['test']))

dict_keys(['train', 'test']) 

{'label': 3, 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)', 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."} 

120000 

7600


In [ ]:
def shift_labels(example):
    example['label'] -= 1
    return example

dataset = dataset.map(shift_labels)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
import huggingface_hub

huggingface_hub.login()

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# model_name = "distilbert/distilbert-base-uncased"
model_name = "google-bert/bert-base-uncased"
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=4).to(device)
tokenizer = BertTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
inputs = tokenizer("He is a good person", return_tensors='pt')

import torch
with torch.no_grad():
    outputs = model(**inputs)

outputs.logits

tensor([[-0.0187, -0.0524, -0.2608, -0.2862]])

Combining title & description into one input (without manually concatenating, because BERT handles it using its special tokens)

In [ ]:
row = dataset['train'][0]
title = row['title']
description = row['description']
label = row['label']

inputs = tokenizer(title, description, return_tensors='pt')
tokenizer.decode(inputs['input_ids'][0])

"[CLS] wall st. bears claw back into the black ( reuters ) [SEP] reuters - short - sellers, wall street ' s dwindling \\ band of ultra - cynics, are seeing green again. [SEP]"

In [ ]:
outputs = model(**inputs, labels=torch.tensor([label]))
print(outputs.logits, '\n')
print(outputs.loss, '\n')

tensor([[-0.1558, -0.1539, -0.6564, -0.2355]], grad_fn=<AddmmBackward0>) 

tensor(1.3414, grad_fn=<NllLossBackward0>) 



In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=1e-5)

In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples['title'],
        examples['description'],
        max_length=128,
        padding='max_length',
        truncation=True
    )

tokenized_data = dataset.map(tokenize_fn)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
def train(loader, epochs=1):
    model.train()

    for e in range(epochs):
        for b, batch in enumerate(loader):
            input_ids = batch['input_ids'].to(device)
            attention_masks = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_masks,
                labels=labels
            )

            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (b + 1) % 100 == 0:
                print(f'Epoch {e + 1}/{epochs} | Batch {b + 1} | Loss {loss}')


In [ ]:
from torch.utils.data import DataLoader
from transformers import DefaultDataCollator

train_loader = DataLoader(tokenized_data['train'], batch_size=32, shuffle=True, collate_fn=DefaultDataCollator())
train(loader=train_loader, epochs=1)

Epoch 1/1 | Batch 100 | Loss 0.3623335361480713
Epoch 1/1 | Batch 200 | Loss 0.1648179292678833
Epoch 1/1 | Batch 300 | Loss 0.12083861231803894
Epoch 1/1 | Batch 400 | Loss 0.1985984444618225
Epoch 1/1 | Batch 500 | Loss 0.08648562431335449
Epoch 1/1 | Batch 600 | Loss 0.18281123042106628
Epoch 1/1 | Batch 700 | Loss 0.04549119621515274
Epoch 1/1 | Batch 800 | Loss 0.04560749977827072
Epoch 1/1 | Batch 900 | Loss 0.09179408848285675
Epoch 1/1 | Batch 1000 | Loss 0.3002784848213196
Epoch 1/1 | Batch 1100 | Loss 0.14198794960975647
Epoch 1/1 | Batch 1200 | Loss 0.05943182110786438
Epoch 1/1 | Batch 1300 | Loss 0.22497501969337463
Epoch 1/1 | Batch 1400 | Loss 0.04532443732023239
Epoch 1/1 | Batch 1500 | Loss 0.30217301845550537
Epoch 1/1 | Batch 1600 | Loss 0.07551328092813492
Epoch 1/1 | Batch 1700 | Loss 0.06949862837791443
Epoch 1/1 | Batch 1800 | Loss 0.1696145087480545
Epoch 1/1 | Batch 1900 | Loss 0.02687578648328781
Epoch 1/1 | Batch 2000 | Loss 0.13852724432945251
Epoch 1/1 | Ba

In [ ]:
model.save_pretrained('news-classifier')
tokenizer.save_pretrained('news-classifier')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('news-classifier/tokenizer_config.json', 'news-classifier/tokenizer.json')

In [ ]:
from huggingface_hub import login

login()

In [ ]:
model.push_to_hub(repo_id='mafgit/news-classifier')
tokenizer.push_to_hub(repo_id='mafgit/news-classifier')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...cgryjk1/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/mafgit/news-classifier/commit/738a9be959d3cfea83ba00a3a838ea99f1ef69cf', commit_message='Upload tokenizer', commit_description='', oid='738a9be959d3cfea83ba00a3a838ea99f1ef69cf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mafgit/news-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='mafgit/news-classifier'), pr_revision=None, pr_num=None)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def evaluate(loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for b, batch in enumerate(loader):
            input_ids = batch['input_ids'].to(device)
            attention_masks = batch['attention_mask'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_masks
            )

            labels = batch['labels'].to(device)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(outputs.logits.argmax(dim=1).cpu().tolist())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f'Accuracy: {acc} | F1-Score: {f1}')

In [ ]:
test_loader = DataLoader(tokenized_data['test'], batch_size=32, shuffle=True, collate_fn=DefaultDataCollator())
evaluate(test_loader)

Accuracy: 0.9453947368421053 | F1-Score: 0.9454663919876926
